# 04 — Extract Features

**Purpose.** Load each trained pilot checkpoint (`03`), run a forward pass
over every split (train/val/calib/test), and save the penultimate-layer
activations — the exact tensor NRC geometry will operate on once `05`'s
formula-confirmation gate clears.

**No NRC-specific math here** — this notebook only produces
"trained model + data in -> features out". The NRC1/NRC2/NRC3 computation
itself stays entirely inside `05`, gated as planned.

**Expected runtime:** well under a minute for all 12 pilot datasets — these
are small models and small datasets; feature extraction is just a forward
pass, no gradients.
**GPU:** used if available, not required.


## Step 1 — Locate project, import helpers

In [ ]:
import sys
from pathlib import Path

if "PROJECT_ROOT" not in dir():
    _here = Path.cwd()
    for candidate in [_here, *_here.parents]:
        if (candidate / "src" / "utils" / "env_utils.py").exists():
            PROJECT_ROOT = candidate
            break
    else:
        raise FileNotFoundError("PROJECT_ROOT not found. Run 00_environment.ipynb first.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import env_utils
from src.models import pilot_mixture_model as pmm
from src.models import feature_extraction as fe

EXTERNAL_DIR = PROJECT_ROOT / "external" / "quantile-recalibration-training"
if not EXTERNAL_DIR.exists():
    print("External repo not found -- cloning now (normally done by notebook 01).")
    env_utils.clone_or_pull_repo(
        repo_url="https://github.com/Vekteur/quantile-recalibration-training.git",
        dest=EXTERNAL_DIR, branch="main",
    )

MixturePrediction = pmm.import_mixture_prediction(PROJECT_ROOT)
pmm.set_mixture_prediction_cls(MixturePrediction)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


## Step 2 — Load prepared splits (`02`) and locate checkpoints (`03`)

In [ ]:
import pickle
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

splits_path = (PATHS["outputs"] if "PATHS" in dir() else PROJECT_ROOT / "outputs") / "uci_pilot_splits.pkl"
if not splits_path.exists():
    raise FileNotFoundError(f"{splits_path} not found -- run 02_prepare_datasets.ipynb first.")
with open(splits_path, "rb") as f:
    saved = pickle.load(f)
all_splits = saved["splits"]

MIXTURE_SIZE = 1  # must match the value used in 03
CKPT_DIR = (PATHS["checkpoints"] if "PATHS" in dir() else PROJECT_ROOT / "checkpoints") / f"mixture_{MIXTURE_SIZE}"
missing = [name for name in all_splits if not (CKPT_DIR / f"{name}.pt").exists()]
if missing:
    raise FileNotFoundError(
        f"No checkpoint for {missing} in {CKPT_DIR} -- run 03_train_pilot_models.ipynb first "
        f"(with MIXTURE_SIZE={MIXTURE_SIZE})."
    )
print(f"Found checkpoints for all {len(all_splits)} datasets in {CKPT_DIR}")


## Step 3 — Extract features for every split of every dataset

In [ ]:
extracted = fe.extract_features_for_all_datasets(CKPT_DIR, all_splits, device=DEVICE)
print(f"\nExtracted features for {len(extracted)} datasets.")


## Step 4 — Sanity checks

Two cheap, independent checks: feature dimensionality must be the verified
hidden width (128) for every dataset regardless of input dimensionality,
and no feature tensor should contain NaN/Inf (which would silently poison
every downstream NRC/PCE computation).

In [ ]:
import numpy as np

problems = []
for name, per_split in extracted.items():
    for split_name, data in per_split.items():
        feats = data["features"]
        if feats.shape[1] != pmm.DEFAULT_HIDDEN_SIZES[-1]:
            problems.append(f"{name}/{split_name}: unexpected feature width {feats.shape[1]}")
        if not np.isfinite(feats).all():
            problems.append(f"{name}/{split_name}: non-finite values in features")

if problems:
    print("[warn] Issues found:")
    for p in problems:
        print(f"  - {p}")
else:
    print(f"All feature tensors: width={pmm.DEFAULT_HIDDEN_SIZES[-1]}, all finite.")

print("\nPer-dataset calibration-split feature summary:")
for name, per_split in extracted.items():
    calib_feats = per_split["calib"]["features"]
    print(f"  {name:10s}  calib features shape={calib_feats.shape}  "
          f"mean_norm={np.linalg.norm(calib_feats, axis=1).mean():.3f}")


## Step 5 — Save extracted features for `05`

In [ ]:
out_path = (PATHS["outputs"] if "PATHS" in dir() else PROJECT_ROOT / "outputs") / f"uci_pilot_features_mixture_{MIXTURE_SIZE}.pkl"
with open(out_path, "wb") as f:
    pickle.dump({"features": extracted, "mixture_size": MIXTURE_SIZE}, f)
print(f"Saved to {out_path}  ({out_path.stat().st_size / 1e6:.2f} MB)")


## Next steps — the formula gate

**Next:** `05_compute_NRC.ipynb`. This is the notebook that will not be
written until the exact NRC1/NRC2/NRC3 formulas are confirmed from:

- Kim, Han & Papyan (or equivalent authors — to be re-confirmed at read
  time), *The Prevalence of Neural Collapse in Neural Multivariate
  Regression*, NeurIPS 2024.
- *Geometric Analysis of Neural Regression Collapse via Intrinsic
  Dimension* (the follow-up paper).

Everything upstream of this point (01-04) is now real, tested, and
verified against source. `05` is where this project's own "never invent
equations, stop if ambiguous" rule actually applies for the first time —
and it will be honored literally: the next step is reading those two
papers closely enough to transcribe their exact formulas, not guessing
a plausible-looking regression analogue of the classification NC1/NC2/NC3
metrics already implemented conceptually for NCCS.
